In [8]:
import requests
import time
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime

API_KEY = "b28c718c-de8b-40f9-b84d-713423720471_cnwnqia"
BASE_URL = "https://server.smartlead.ai/api/v1"

DB_CONFIG = {
    "host": "gw-rds-analytics.celzx4qnlkfp.us-east-1.rds.amazonaws.com",
    "port": 5432,
    "dbname": "gw_prod",
    "user": "airbyte_user",
    "password": "airbyte_user_password"
}

LIMIT = 100

In [9]:
def get_filtered_campaign_ids():
    url = f"{BASE_URL}/campaigns"

    params = {
        "api_key": API_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    campaigns = response.json()

    filtered_ids = []

    for c in campaigns:
        name = (c.get("name") or "").lower()

        if "manufacturing" in name or "e004" in name:
            filtered_ids.append(c["id"])

    print(f"Total campaigns fetched: {len(campaigns)}")
    print(f"Filtered campaigns: {len(filtered_ids)}")

    return filtered_ids

In [10]:
CAMPAIGN_IDS = get_filtered_campaign_ids()
print(CAMPAIGN_IDS)


Total campaigns fetched: 68
Filtered campaigns: 20
[3170717, 3170697, 3170652, 3103614, 3103594, 3103585, 3103535, 3103483, 3103452, 3103304, 3088976, 3083406, 3083390, 3083336, 3083047, 3082596, 3027655, 2902041, 2853186, 2811964]


In [11]:
from datetime import datetime, timedelta

# current UTC time
now = datetime.utcnow()

# today at 00:00 UTC
today_utc_midnight = now.replace(hour=0, minute=0, second=0, microsecond=0)

# 🔥 last 7 FULL days window (excluding today)
start = today_utc_midnight - timedelta(days=7)
end = today_utc_midnight

# format in ISO (same as your previous format)
START_DATE = start.strftime('%Y-%m-%dT%H:%M:%S')
END_DATE = end.strftime('%Y-%m-%dT%H:%M:%S')

print("START_DATE:", START_DATE)
print("END_DATE:", END_DATE)

START_DATE: 2026-04-08T00:00:00
END_DATE: 2026-04-15T00:00:00


/tmp/ipykernel_671/3643477048.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


In [5]:
def safe_request(url, params, retries=5):
    for i in range(retries):
        response = requests.get(url, params=params)

        if response.status_code == 200:
            return response

        if response.status_code == 429:
            wait = min(2 ** i, 30)
            print(f"⚠️ Rate limited. Sleeping {wait}s")
            time.sleep(wait)
            continue

        raise Exception(f"Error {response.status_code}: {response.text}")

    raise Exception("Max retries exceeded")

In [6]:
def process_campaign(campaign_id, conn):
    print(f"\n🚀 Processing campaign {campaign_id}")

    cur = conn.cursor()

    # -------------------------
    # FETCH LEADS
    # -------------------------
    offset = 0

    while True:
        url = f"{BASE_URL}/campaigns/{campaign_id}/leads"

        params = {
            "api_key": API_KEY,
            "limit": LIMIT,
            "offset": offset
        }

        response = safe_request(url, params)
        data = response.json().get("data", [])

        if not data:
            break

        rows = []
        rows_map = {}

        for r in data:
            lead = r.get("lead", {})
            email = (lead.get("email") or "").strip().lower()

            if not email:
                continue

            #dedupe key
            key = email

            rows_map[key] = (
                lead.get("id"),
                lead.get("first_name"),
                lead.get("last_name"),
                email,
                lead.get("phone_number"),
                lead.get("company_name"),
                lead.get("website"),
                lead.get("location"),
                lead.get("linkedin_profile"),
                lead.get("company_url"),
                lead.get("is_unsubscribed")
            )

        rows = list(rows_map.values())
        # for r in data:
        #     lead = r.get("lead", {})

        #     email = (lead.get("email") or "").strip().lower()
        #     if not email:
        #         continue

        #     rows.append((
        #         lead.get("id"),
        #         lead.get("first_name"),
        #         lead.get("last_name"),
        #         email,
        #         lead.get("phone_number"),
        #         lead.get("company_name"),
        #         lead.get("website"),
        #         lead.get("location"),
        #         lead.get("linkedin_profile"),
        #         lead.get("company_url"),
        #         lead.get("is_unsubscribed")
        #     ))

        # if rows:
        #     execute_values(cur, """
        #         INSERT INTO gist.gtm_smartlead_leads (
        #             lead_id, lead_first_name, lead_last_name, lead_email,
        #             lead_phone_number, lead_company_name, lead_website,
        #             lead_location, linkedin_profile, company_url, is_unsubscribed
        #         ) VALUES %s
        #         ON CONFLICT (lead_id) DO UPDATE SET
        #             lead_email = EXCLUDED.lead_email,
        #             lead_phone_number = EXCLUDED.lead_phone_number
        #     """, rows, page_size=200)
        #     conn.commit()

        if rows:
          rows.sort(key=lambda x: x[0])  #CRITICAL

          for attempt in range(3):
              try:
                  execute_values(
                      cur,
                      """
                      INSERT INTO gist.gtm_smartlead_leads (
                          lead_id, lead_first_name, lead_last_name, lead_email,
                          lead_phone_number, lead_company_name, lead_website,
                          lead_location, linkedin_profile, company_url, is_unsubscribed
                      ) VALUES %s
                      ON CONFLICT (lead_id) DO UPDATE SET
                          lead_email = EXCLUDED.lead_email,
                          lead_phone_number = EXCLUDED.lead_phone_number
                      """,
                      rows,
                      page_size=200
                  )

                  conn.commit()
                  break

              except Exception as e:
                  if "deadlock" in str(e).lower():
                      print(f"⚠️ Deadlock retry {attempt+1}")
                      conn.rollback()
                      time.sleep(1 * (attempt + 1))
                  else:
                      raise e

        offset += LIMIT

    print(f"✅ Leads done for {campaign_id}")

    # -------------------------
    # FETCH STATS
    # -------------------------
    offset = 0

    while True:
        url = f"{BASE_URL}/campaigns/{campaign_id}/statistics"

        params = {
            "api_key": API_KEY,
            "limit": 1000,
            "offset": offset,
            "sent_time_start_date": START_DATE,
            "sent_time_end_date": END_DATE
        }

        response = safe_request(url, params)
        data = response.json().get("data", [])

        if not data:
            break

        rows = []
        for r in data:
            rows.append((
                r.get("stats_id"),
                campaign_id,
                r.get("email_campaign_seq_id"),
                r.get("sequence_number"),
                r.get("lead_name"),
                r.get("lead_email"),
                r.get("lead_category"),
                r.get("seq_variant_id"),
                r.get("email_subject"),
                r.get("email_message"),
                r.get("sent_time"),
                r.get("open_time"),
                r.get("click_time"),
                r.get("reply_time"),
                r.get("open_count", 0),
                r.get("click_count", 0),
                r.get("is_unsubscribed", False),
                r.get("is_bounced", False),
                datetime.utcnow()
            ))

        if rows:
            execute_values(cur, """
                INSERT INTO gist.gtm_email_campaign_stats (
                    stats_id, campaign_id, email_campaign_seq_id,
                    sequence_number, lead_name, lead_email,
                    lead_category, seq_variant_id, email_subject,
                    email_message, sent_time, open_time,
                    click_time, reply_time, open_count,
                    click_count, is_unsubscribed, is_bounced, created_at
                ) VALUES %s
                ON CONFLICT (stats_id) DO UPDATE SET
                    open_count = EXCLUDED.open_count,
                    click_count = EXCLUDED.click_count
            """, rows, page_size=200)
            conn.commit()

        offset += 1000

    print(f"✅ Stats done for {campaign_id}")

    conn.commit()

In [7]:
conn = psycopg2.connect(**DB_CONFIG)

for cid in CAMPAIGN_IDS:
    process_campaign(cid, conn)

conn.close()

print("\n🎯 All campaigns processed successfully")


🚀 Processing campaign 3170717
✅ Leads done for 3170717


/tmp/ipykernel_671/2524112878.py:167: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()


✅ Stats done for 3170717

🚀 Processing campaign 3170697
✅ Leads done for 3170697
✅ Stats done for 3170697

🚀 Processing campaign 3170652
✅ Leads done for 3170652
✅ Stats done for 3170652

🚀 Processing campaign 3103614
✅ Leads done for 3103614
✅ Stats done for 3103614

🚀 Processing campaign 3103594
✅ Leads done for 3103594
✅ Stats done for 3103594

🚀 Processing campaign 3103585
✅ Leads done for 3103585
✅ Stats done for 3103585

🚀 Processing campaign 3103535
✅ Leads done for 3103535
✅ Stats done for 3103535

🚀 Processing campaign 3103483
✅ Leads done for 3103483
✅ Stats done for 3103483

🚀 Processing campaign 3103452
✅ Leads done for 3103452
✅ Stats done for 3103452

🚀 Processing campaign 3103304
✅ Leads done for 3103304
✅ Stats done for 3103304

🚀 Processing campaign 3088976
✅ Leads done for 3088976
✅ Stats done for 3088976

🚀 Processing campaign 3083406
✅ Leads done for 3083406
✅ Stats done for 3083406

🚀 Processing campaign 3083390
✅ Leads done for 3083390
✅ Stats done for 3083390

🚀 